# Code Mode Agents - Data Analysis

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/code-mode-analysis/code-mode-analysis-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/code-mode-analysis
    !uv pip install -r requirements.txt
    !uv pip install keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

from utils.file_viewer import view_file

In [ ]:
# Set the Anthropic key. Skip this if it's already in a .env or your environment —
# config.py calls load_dotenv() for you.
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

In [ ]:
# set Flyte cluster

Runing Flyte workflows

--local
--tui

Local runs still get caching, retries, TUI

remote


Fanout
scale
containerized
UI


### 0. Download the dataset (optional, but do it before a workshop)

In [ ]:
!flyte run --local step0_download_data.py download

### 1. The sandbox, with no LLM anywhere

In [ ]:

!flyte run --local step1_sandbox.py monthly_tip_trend

### 2. The model writes the program

In [ ]:
!flyte run --local step2_generated_code.py analyze \
    --question "Did tipping change between January and December 2024?"

In [ ]:
# The task was decorated @env.task(report=True), so it wrote an HTML report.
# Render it here — charts, metrics, and the program the model actually wrote.
from report import show_latest

show_latest()

### 3. The agent, and the fan-out

`Agent(code_mode=True)` writes one program, and `query` is an `@env.task` — so every query the model writes dispatches as a **durable child task**. Look for `flyte_map("query", sqls, months, concurrency=4)` in the program it wrote below.

> **Note:** running `--local`, that `flyte_map` executes **sequentially** — Flyte warns you so in the output. The generated code and the answer are identical, but the twelve-parallel-containers payoff only happens on a cluster. Run this one remotely (drop `--local`) and open the run in the UI to see the child tasks.

In [ ]:
!flyte run --local step3_agent_report.py analyze \
    --question "Rank the boroughs by tip rate and show how it moved through 2024"

In [ ]:
from report import show_latest

show_latest()

### 4. Why bother? Measure it.

In [ ]:
!flyte run --local step4_compare_modes.py compare \
    --question "Which borough tipped best in each quarter of 2024?"

In [ ]:
# Turns and tokens, side by side: sequential tool calling vs code mode.
from report import show_latest

show_latest()

### 5. Serve it as a chat app (stretch)

The whole web layer is one declaration — `AgentChatAppEnvironment` brings the chat UI, the tools sidebar, streaming, and the endpoint.

The cell below runs it **inside the notebook**: uvicorn in a background thread, and Colab proxies the port into an iframe. Ask it something, and it writes a program to answer you.

Things to try:
- *"Do riders in Brooklyn and the Bronx really tip less than Manhattan, or is something else going on?"*
- *"How did the tip rate move month by month through 2024? Chart it."*
- Then a follow-up, since the conversation is memory: *"Now compare those to the shortest trips."*

Deploying it (the last cell) is what makes each message a **durable run** — queries fan out as child tasks you can click into. Running it here in the notebook does not: the agent runs in this process.

In [ ]:
# Serve the chat app inside the notebook.
#
# uvicorn runs in a background thread so the cell doesn't block, then Colab proxies
# the port into an iframe below. No tunnel, no ngrok token.
#
# This is the *local* shape of the app: the agent runs in this process, so there are
# no durable child tasks and no real fan-out (see step 3). It's the right way to play
# with the prompt and the UI; deploy it (next cell) for the durable version.
import threading

import uvicorn

import flyte
from step5_chat_app import chat_env

flyte.init()  # local: tasks run in-process

PORT = 8080
server = uvicorn.Server(
    uvicorn.Config(chat_env(local=True).build_fastapi_app(),
                   host="0.0.0.0", port=PORT, log_level="warning")
)
threading.Thread(target=server.run, daemon=True).start()

if IN_COLAB:
    from google.colab import output

    output.serve_kernel_port_as_iframe(PORT, height=900)
else:
    print(f"NYC Taxi analyst → http://localhost:{PORT}")

In [ ]:
# Deploy it to the cluster instead — needs a Flyte/Union connection.
!python step5_chat_app.py deploy